# GAMMA DNA Tutorial 2 — Telco Churn Multi-Classifier

Replicating the FACET *Scikit-learn Classifier Summaries* tutorial using **GAMMA_DNA**.

| | |
|---|---|
| **Dataset** | Kaggle Telco Customer Churn (`KAGGLE-Telco-Customer-Churn.csv`) |
| **Target** | `Churn` (1 = churned, 0 = retained) |
| **Focus** | Multi-model comparison, synergy/redundancy, tenure simulation |


In [1]:
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DAPS_BRIX = "C:/Users/Dai Dennis/OneDrive - The Boston Consulting Group, Inc/Desktop/DAPS_Brix"
if DAPS_BRIX not in sys.path:
    sys.path.insert(0, DAPS_BRIX)

BASE_URL = (
    "https://raw.githubusercontent.com/BCG-X-Official/facet/"
    "aada69350b085f8a3dae7900b16db6c79b146f60/sphinx/source/tutorial/"
)

df = pd.read_csv(BASE_URL + "KAGGLE-Telco-Customer-Churn.csv")
print(f"Shape: {df.shape}")
df.head()


Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Data Preprocessing

- Drop `customerID` (non-predictive)
- Fix `TotalCharges` (stored as string due to empty entries)
- Encode all categorical columns to numeric


In [2]:
from sklearn.preprocessing import LabelEncoder

# Drop ID column
df.drop(columns=["customerID"], inplace=True)

# Fix TotalCharges (whitespace string → NaN → fill with median)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

# Encode target
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Encode simple Yes/No binary columns
for col in ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]:
    df[col] = df[col].map({"Yes": 1, "No": 0})

# Encode gender
df["gender"] = df["gender"].map({"Male": 1, "Female": 0})

# Label-encode all remaining string columns
le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col].astype(str))

print(f"Target distribution:\n{df['Churn'].value_counts()}\n")
print(f"All-numeric: {df.select_dtypes(include='object').empty}")
df.dtypes


Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

All-numeric: True


gender                int64
SeniorCitizen         int64
Partner               int64
Dependents            int64
tenure                int64
PhoneService          int64
MultipleLines         int32
InternetService       int32
OnlineSecurity        int32
OnlineBackup          int32
DeviceProtection      int32
TechSupport           int32
StreamingTV           int32
StreamingMovies       int32
Contract              int32
PaperlessBilling      int64
PaymentMethod         int32
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

## Exploratory Data Analysis


In [3]:
import plotly.express as px

fig = px.histogram(
    df, x="tenure", color="Churn",
    title="Tenure Distribution by Churn Status",
    template="plotly_dark",
    barmode="overlay", opacity=0.75,
    labels={"tenure": "Months with company", "Churn": "Churn (1=Yes)"},
)
fig.show()


In [4]:
fig2 = px.box(
    df, x="Churn", y="MonthlyCharges",
    title="Monthly Charges vs Churn",
    template="plotly_dark",
    labels={"Churn": "Churn (1=Yes)", "MonthlyCharges": "Monthly Charges ($)"},
)
fig2.show()


## Multi-Model Comparison

Train Logistic Regression, Random Forest, and Gradient Boosting; compare
test AUC and 5-fold cross-validation scores.


In [5]:
from gamma.pipeline import GAMMA_DNA

g = GAMMA_DNA(df, target="Churn")
comparison = g.compare_models()
comparison


,model_type,train_score,test_score,cv_mean,cv_std,metric
0,logistic_regression,0.8428,0.8614,0.8396,0.0111,roc_auc
1,random_forest_classifier,0.9999,0.8319,0.8232,0.0122,roc_auc
2,gradient_boosting_classifier,0.8791,0.8582,0.8412,0.0098,roc_auc


## Train Best Model — Gradient Boosting Classifier


In [6]:
result = g.train(model_type="gradient_boosting_classifier")
result.summary()



  ModelResult: gradient_boosting_classifier  [binary_classification]
  Metric                                Train         Test
  --------------------------------------------------------
  accuracy                             0.8293       0.8013
  f1                                   0.8220       0.7918
  precision                            0.8218       0.7904
  recall                               0.8293       0.8013
  roc_auc                              0.8814       0.8449


In [7]:
result.plot()
print(f"\nTest AUC  : {result.roc_auc:.4f}")
print(f"Accuracy  : {result.accuracy:.4f}")
print(f"F1 Score  : {result.f1:.4f}")
print(f"Precision : {result.precision:.4f}")
print(f"Recall    : {result.recall:.4f}")



Test AUC  : 0.8449
Accuracy  : 0.8013
F1 Score  : 0.7918
Precision : 0.7904
Recall    : 0.8013


## SHAP Feature Importance


In [8]:
report = g.explain(compute_shap=True, compute_permutation=False)
report.plot_overview(max_display=15)


  Computing SHAP values on 19 features… (may take 10-60s)
  Done.


In [9]:
imp = report.to_frame()
imp.sort_values("score", ascending=False).head(15)


,model_imp,perm_imp_mean,shap_mean_abs,rank,score
feature,,,,,
Contract,0.403882,NaN,0.760001,1,0.760001
tenure,0.142749,NaN,0.366153,2,0.366153
MonthlyCharges,0.134967,NaN,0.335264,3,0.335264
OnlineSecurity,0.081398,NaN,0.261517,4,0.261517
TechSupport,0.054091,NaN,0.222961,5,0.222961
OnlineBackup,0.010986,NaN,0.127596,6,0.127596
TotalCharges,0.078269,NaN,0.125068,7,0.125068
PaperlessBilling,0.020697,NaN,0.124807,8,0.124807
PaymentMethod,0.023649,NaN,0.107328,9,0.107328


## Feature Synergy Analysis


In [10]:
report.plot_synergy(kind="both", sample_size=200)


## Feature Redundancy Analysis


In [11]:
report.plot_redundancy(kind="both")


## Partial Dependence — Tenure & Monthly Charges


In [12]:
report.plot_pdp("tenure", grid_points=30)


In [13]:
report.plot_pdp("MonthlyCharges", grid_points=30)


## Simulation — Tenure Impact on Churn Probability

How does churn probability change as customers stay longer?


In [14]:
tenure_grid = np.arange(1, 73, 4)
report.plot_simulate("tenure", tenure_grid)


In [15]:
charges_grid = np.linspace(
    df["MonthlyCharges"].min(),
    df["MonthlyCharges"].max(),
    25,
)
report.plot_simulate("MonthlyCharges", charges_grid)


## Top SHAP Feature Interactions


In [16]:
report.plot_top_interactions(top_k=10, sample_size=200)


In [17]:
top_int = report.top_interactions(top_k=10, sample_size=200)
top_int


,feature_a,feature_b,mean_abs_interaction
0,tenure,Contract,0.064357
1,Contract,MonthlyCharges,0.037135
2,tenure,MonthlyCharges,0.035230
3,MonthlyCharges,TotalCharges,0.034759
4,TechSupport,MonthlyCharges,0.027502
5,OnlineSecurity,Contract,0.027162
6,Contract,TotalCharges,0.024167
7,tenure,TotalCharges,0.024041
8,TechSupport,Contract,0.023343
9,tenure,PaperlessBilling,0.023049


## Facet View — Redundancy & Synergy Side-by-Side


In [18]:
report.plot_facet(sample_size=200)


## Redundancy Linkage Chart


In [19]:
report.plot_redundancy(kind="linkage")
